# _blocks 와 비용 흐름 체감하기

두 번째 탐색 노트북. (입력 클래스는 `explore_scheduler.ipynb` 참고)

여기서 보는 것:
- `_blocks` 가 outer 루프를 펼치며 내놓는 **5-튜플** 이 실제로 어떻게 생겼나
- **mixed-precision** 이면 블록마다 W/compute 가 어떻게 달라지나 (핵심 체감)
- 그 5-튜플이 `footprint_bits` / `dram_bits` 로 어떻게 흘러가나

각 `### 바꿔보기` 셀에서 `wbits` / `perm` / `m_in,k_in,n_in` 만 고치고 Shift+Enter.

In [1]:
import os, sys, importlib
sys.path.insert(0, os.path.abspath("."))
import mxp_scheduler as s
importlib.reload(s)

# 5-튜플을 보기 좋은 표로 출력하는 헬퍼
def show_blocks(m, w, title=""):
    out, inn = s._out_in(m, w)
    print(f"{title}")
    print(f"perm={''.join(m.perm)}  inn={inn}  out={out}  -> 블록수 {out['M']*out['K']*out['N']}")
    print(f"{'#':>2} {'idx (M,K,N)':<16}{'compute_blk':>12}{'a_blk':>9}{'w_blk':>9}{'c_blk':>9}")
    for i, (idx, comp, a_blk, w_blk, c_blk) in enumerate(s._blocks(m, w)):
        trip = f"({idx['M']},{idx['K']},{idx['N']})"
        print(f"{i:>2} {trip:<16}{comp:>12}{a_blk:>9}{w_blk:>9}{c_blk:>9}")
    print()

print("loaded:", s.__file__)

loaded: c:\Users\ptj72\Desktop\Desktop\00project\gemm_sram\MXP_scheduler\mxp_scheduler.py


## A. _blocks 가 내놓는 5-튜플

한 블록 = `(idx, compute_blk, a_blk, w_blk, c_blk)`

| 위치 | 이름 | 뜻 |
|---|---|---|
| [0] | `idx` | 지금 outer 루프 위치 `{M,K,N}` |
| [1] | `compute_blk` | 이 블록 계산량 = `32 * n_in * (상주 wbits 합)` |
| [2] | `a_blk` | A 조각 비트 (모든 블록 동일) |
| [3] | `w_blk` | W 조각 비트 (mixed면 블록마다 다름) |
| [4] | `c_blk` | C 조각 비트 (모든 블록 동일) |

먼저 **균일 8비트** -> 모든 블록이 똑같이 나온다.

In [2]:
### 바꿔보기 (균일)
w = s.Work(M=128, K=128, N=128, wbits=[[8]*4 for _ in range(4)], act_bits=8)
m = s.Mapping(perm=("M","K","N"), m_in=1, k_in=2, n_in=4)
show_blocks(m, w, "[균일 8-bit] 모든 블록 동일해야 정상")
print("compute_blk 합 =", sum(b[1] for b in s._blocks(m, w)),
      " == compute_work =", s.compute_work(w), " (일치하면 OK)")

[균일 8-bit] 모든 블록 동일해야 정상
perm=MKN  inn={'M': 1, 'K': 2, 'N': 4}  out={'M': 4, 'K': 2, 'N': 1}  -> 블록수 8
 # idx (M,K,N)      compute_blk    a_blk    w_blk    c_blk
 0 (0,0,0)                 2048    65536    16384   131072
 1 (0,1,0)                 2048    65536    16384   131072
 2 (1,0,0)                 2048    65536    16384   131072
 3 (1,1,0)                 2048    65536    16384   131072
 4 (2,0,0)                 2048    65536    16384   131072
 5 (2,1,0)                 2048    65536    16384   131072
 6 (3,0,0)                 2048    65536    16384   131072
 7 (3,1,0)                 2048    65536    16384   131072

compute_blk 합 = 16384  == compute_work = 16384  (일치하면 OK)


## B. mixed-precision -> 블록마다 W/compute 가 달라진다 (핵심)

`m_in=1, k_in=1` 로 두면 **한 블록 = 한 타일** 이라, 각 타일의 비트수가 `w_blk` 에 그대로 보인다.
(`w_blk = 그 타일 평균비트 * 1024`,  `compute_blk = 32 * n_in * 그 비트`)

아래 wbits 는 행마다 정밀도를 다르게 줬다 (row0=2bit, row1=4bit, row2=8bit, row3=섞임).

In [3]:
### 바꿔보기 (mixed) - wbits 를 마음대로 고쳐보세요 (각 값 2~8)
wbits = [[2, 2, 2, 2],
         [4, 4, 4, 4],
         [8, 8, 8, 8],
         [2, 8, 2, 8]]
w = s.Work(M=128, K=128, N=128, wbits=wbits, act_bits=8)
m = s.Mapping(perm=("M","K","N"), m_in=1, k_in=1, n_in=4)   # 한 블록 = 한 타일
show_blocks(m, w, "[mixed] w_blk 가 타일 비트수따라 2048/4096/8192 로 달라짐")

wmax = max(b[3] for b in s._blocks(m, w))
wmin = min(b[3] for b in s._blocks(m, w))
print(f"w_blk 최소={wmin}  최대={wmax}")
print(f"footprint 는 최대값({wmax}) 기준으로 SRAM 예약 -> 저비트 블록 땐 {wmax-wmin} bit 놀고있음")
print("  (== 우리가 메모리에 적은 precision-adaptive blocking 필수수정 포인트)")

[mixed] w_blk 가 타일 비트수따라 2048/4096/8192 로 달라짐
perm=MKN  inn={'M': 1, 'K': 1, 'N': 4}  out={'M': 4, 'K': 4, 'N': 1}  -> 블록수 16
 # idx (M,K,N)      compute_blk    a_blk    w_blk    c_blk
 0 (0,0,0)                  256    32768     2048   131072
 1 (0,1,0)                  256    32768     2048   131072
 2 (0,2,0)                  256    32768     2048   131072
 3 (0,3,0)                  256    32768     2048   131072
 4 (1,0,0)                  512    32768     4096   131072
 5 (1,1,0)                  512    32768     4096   131072
 6 (1,2,0)                  512    32768     4096   131072
 7 (1,3,0)                  512    32768     4096   131072
 8 (2,0,0)                 1024    32768     8192   131072
 9 (2,1,0)                 1024    32768     8192   131072
10 (2,2,0)                 1024    32768     8192   131072
11 (2,3,0)                 1024    32768     8192   131072
12 (3,0,0)                  256    32768     2048   131072
13 (3,1,0)                 1024    32768     819

## C. 5-튜플 -> footprint_bits (책상 넓이)

`footprint = a_blk + max(w_blk) + c_blk` (A조각 + 가장무거운 W조각 + C조각).
직접 세 조각으로 분해해서 함수값과 맞춰본다.

In [4]:
blocks = list(s._blocks(m, w))
a_blk = blocks[0][2]
c_blk = blocks[0][4]
foot_w = max(b[3] for b in blocks)   # W 만 max

print(f"A 조각 a_blk         = {a_blk:>8}  (첫 블록값, 모든 블록 동일)")
print(f"W 조각 max(w_blk)    = {foot_w:>8}  (가장 무거운 블록 기준)")
print(f"C 조각 c_blk         = {c_blk:>8}  (첫 블록값, 모든 블록 동일)")
print(f"                       --------")
print(f"합계                 = {a_blk+foot_w+c_blk:>8}")
print(f"footprint_bits()     = {s.footprint_bits(m, w):>8}  (일치 확인)")

hw = s.HW(bank_size=1024, banks=32, dram_bw=64)
print(f"\ncap_bits = {hw.cap_bits} -> feasible? {s.feasible(m, w, hw)}")

A 조각 a_blk         =    32768  (첫 블록값, 모든 블록 동일)
W 조각 max(w_blk)    =     8192  (가장 무거운 블록 기준)
C 조각 c_blk         =   131072  (첫 블록값, 모든 블록 동일)
                       --------
합계                 =   172032
footprint_bits()     =   172032  (일치 확인)

cap_bits = 1048576 -> feasible? True


## D. 5-튜플 -> dram_bits (트래픽) 의 직관

`dram_bits` 는 블록을 순서대로 걸으면서 **인덱스가 바뀔 때만** 재로드를 센다:
- A 는 `(K,N)` 이 바뀌면 재로드  (A 는 [K,N] 텐서)
- W 는 `(M,K)` 가 바뀌면 재로드  (W 는 [M,K] 텐서)

아래는 그 판정을 블록마다 직접 표시한 것 (dram_bits 내부와 동일 로직).
어떤 블록에서 A/W 가 다시 로드되는지 ✓ 로 보인다.

In [ ]:
def show_reloads(m, w):
    print(f"perm={''.join(m.perm)}")
    print(f"{'#':>2} {'idx(M,K,N)':<14}{'A재로드':>8}{'W재로드':>8}  (A:(K,N)변화 / W:(M,K)변화)")
    prev = None
    A = W = 0
    for i, (idx, comp, a_blk, w_blk, c_blk) in enumerate(s._blocks(m, w)):
        a_re = prev is None or (idx['K'],idx['N']) != (prev['K'],prev['N'])
        w_re = prev is None or (idx['M'],idx['K']) != (prev['M'],prev['K'])
        A += a_blk if a_re else 0
        W += w_blk if w_re else 0
        trip = f"({idx['M']},{idx['K']},{idx['N']})"
        print(f"{i:>2} {trip:<14}{('YES' if a_re else '.'):>8}{('YES' if w_re else '.'):>8}")
        prev = idx
    print(f"-> A 트래픽합={A}  W 트래픽합={W}")
    d = s.dram_bits(m, w)
    print(f"-> dram_bits(): A={d['A']}  W={d['W']}  Cw={d['Cw']}  Cr={d['Cr']}  (위 합과 A/W 일치 확인)")
    print()

# 같은 GEMM 을 두 루프순서로 비교 -> 트래픽이 달라지는 걸 직접 본다
show_reloads(s.Mapping(perm=("M","K","N"), m_in=1, k_in=1, n_in=4), w)
show_reloads(s.Mapping(perm=("N","K","M"), m_in=1, k_in=1, n_in=4), w)

perm=MKN
 # idx(M,K,N)        A재로드    W재로드  (A:(K,N)변화 / W:(M,K)변화)
 0 (0,0,0)            YES     YES
 1 (0,1,0)            YES     YES
 2 (0,2,0)            YES     YES
 3 (0,3,0)            YES     YES
 4 (1,0,0)            YES     YES
 5 (1,1,0)            YES     YES
 6 (1,2,0)            YES     YES
 7 (1,3,0)            YES     YES
 8 (2,0,0)            YES     YES
 9 (2,1,0)            YES     YES
10 (2,2,0)            YES     YES
11 (2,3,0)            YES     YES
12 (3,0,0)            YES     YES
13 (3,1,0)            YES     YES
14 (3,2,0)            YES     YES
15 (3,3,0)            YES     YES
-> A 트래픽합=524288  W 트래픽합=77824
-> dram_bits(): A=524288.0  W=77824.0  Cw=524288  Cr=0  (위 합과 A/W 일치 확인)

perm=NKM
 # idx(M,K,N)        A재로드    W재로드  (A:(K,N)변화 / W:(M,K)변화)
 0 (0,0,0)            YES     YES
 1 (1,0,0)              .     YES
 2 (2,0,0)              .     YES
 3 (3,0,0)              .     YES
 4 (0,1,0)            YES     YES
 5 (1,1,0)              .     YES
 6 (2,1,0) 

## E. 직접 실험거리 (선택)

- `perm` 을 6가지로 바꿔가며 `show_reloads` -> 어떤 순서가 A/W 재로드를 줄이나?
- `k_in` 을 키우면(상주 늘림) W 재로드가 어떻게 변하나?
- mixed `wbits` 에서 고비트 타일이 자주 재로드되는 순서 vs 적게 재로드되는 순서 비교
  -> 이게 "고비트 타일 오래 상주" 전략을 손으로 확인하는 길 (현재 모델은 자동으론 못함)

In [ ]:
# 6가지 perm 의 W 트래픽 비교 (mixed wbits 에서 고비트 재로드 영향 보기)
import itertools
for perm in itertools.permutations(("M","K","N")):
    mm = s.Mapping(perm=perm, m_in=1, k_in=1, n_in=4)
    d = s.dram_bits(mm, w)
    print(f"perm={''.join(perm)}  W트래픽={d['W']:>8}  A트래픽={d['A']:>8}  total={d['total']:>10.0f}")